## 结构化输出

#### 加载环境并初始化

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

In [ ]:
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from typing import Optional, List, Literal

llm = init_chat_model(
    model='deepseek:deepseek-flash',
    # 需关闭思考模型
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

#### Pydantic

In [9]:
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

structured_llm = llm.with_structured_output(Person)
resp = structured_llm.invoke("张三是一名30岁的软件工程师")
print(resp)

name='张三' age=30 occupation='软件工程师'


In [10]:
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影名")
    year: int = Field(description="发行年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

structured_llm = llm.with_structured_output(Movie)
resp = structured_llm.invoke("给出电影盗梦空间的信息")
print(resp)

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=9.4


#### 高级特效

##### 1、可选字段

如果不存在则为 `None`

In [11]:
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

structured_llm = llm.with_structured_output(Person)
resp = structured_llm.invoke("张三是一名软件工程师")
print(resp)

name='张三' age=None occupation='软件工程师'


##### 2、默认值

不同的模型供应商，对于此字段的支持是不同的，有的不支持

In [12]:
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(default=20, description="年龄")
    occupation: str = Field(description="职业")

structured_llm = llm.with_structured_output(Person)
resp = structured_llm.invoke("张三是一名软件工程师")
print(resp)

name='张三' age=20 occupation='软件工程师'


##### 3、枚举类型

In [13]:
from enum import Enum

class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class Task(BaseModel):
    """任务信息"""
    title: str = Field(description="任务名")
    priority: Priority = Field(description="优先级")
    # priority: Literal["低", "中", "高"] = Field(description="优先级")

structured_llm = llm.with_structured_output(Task)
resp = structured_llm.invoke("紧急完成公司介绍 PPT 的制作")
print(resp)

title='紧急完成公司介绍 PPT 的制作' priority=<Priority.HIGH: '高'>


##### 4、列表提取

In [14]:
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")

class PersonList(BaseModel):
    """任务列表"""
    people: List[Person]

structured_llm = llm.with_structured_output(PersonList)
resp = structured_llm.invoke("张三 30 岁，李四 40 岁")
print(resp)

people=[Person(name='张三', age=30), Person(name='李四', age=40)]


In [15]:
class Review(BaseModel):
    """产品评论"""
    name: str = Field(description="产品名称")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

structured_llm = llm.with_structured_output(Review)
resp = structured_llm.invoke("苹果手机非常棒！摄像头强大，手感好，但是价格贵，没有充电器。")
print(resp)

name='苹果手机' pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']


##### 5、嵌套结构

考虑 LLM 能力有限，复杂嵌套结构可能会出错，建议小于等于三层

In [21]:
class Address(BaseModel):
    """地址信息"""
    city: str = Field(description="城市")
    district: str = Field(description="区域")

class Company(BaseModel):
    """公司信息"""
    name: str = Field(description="公司名称")
    address: Address = Field(description="地址")


structured_llm = llm.with_structured_output(Company)
resp = structured_llm.invoke("阿里巴巴在杭州的滨江区")
print(resp)

name='阿里巴巴' address=Address(city='杭州', district='滨江区')


##### 6、限制条件

不同 LLM 表现不一定一样，有的是将异常值替换为一个正常值

In [20]:
class Product(BaseModel):
    """产品信息"""
    name: str = Field(description="产品名称", min_length=2, max_length=20)
    price: float = Field(description="价格", gt=0)
    stock: int = Field(description="库存", gt=0)


structured_llm = llm.with_structured_output(Product)
try:
    resp = structured_llm.invoke("iphone 18 pro max 的价格是 -10000，当前库存 8000")
    print(resp)
except Exception as e:
    print(e)

1 validation error for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-10000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than
